# Universal Multi-Stock LSTM / Transformer (Full Pipeline)

Trains a single model on multiple symbols using the full feature set:
technical indicators + FinBERT sentiment embeddings + fundamental data.

**Prerequisites (run once before this notebook):**
- Market data: run the *Save stock price to cache* section of `runML.ipynb`
  with `timeframe = "1Day"` — this notebook requires daily bars.
  If hourly data is already cached it will be detected and replaced automatically.
- Fundamentals: fetched automatically below via yfinance (no API key needed).
- News articles: optional. Run `NewsPipeline` or `KaggleImporter` to populate
  `ArticleRepository`. Without articles the sentiment branch uses zero vectors
  (the model still trains, just without a sentiment signal).

**Flow:**
1. Ensure daily OHLCV bars are in `MarketDataCache`
2. Fetch fundamentals from yfinance if not already cached
3. Load or compute daily-aggregated sentiment (cached to `data/sentiment_daily.pkl`)
4. Build fused dataset per symbol (`build_dataset`), pool across symbols
5. Chronological train / val / test split + scaling (`make_loaders`)
6. Train `SentimentLSTM` or `SentimentTransformer`
7. Bootstrap evaluation (AUC + 95 % CI)
8. Live inference on a single stock

In [2]:
from __future__ import annotations

import numpy as np
import pandas as pd
import torch
from pathlib import Path

from sentiment.log import setup_logging
from sentiment.sources.alpaca import AlpacaSource
from sentiment.sources.cache import MarketDataCache
from sentiment.sources.fundamental import FundamentalSource, FundamentalCache
from sentiment.sources.news.repository import ArticleRepository
from sentiment.embeddings.pipeline import SentimentPipeline
from sentiment.features.technical import TechnicalFactors
from sentiment.features.dataloader import FusedDataset, build_dataset, make_loaders
from sentiment.model.lstm import SentimentLSTM
from sentiment.model.transformer import SentimentTransformer
from sentiment.model.train import train_model, bootstrap_evaluate

setup_logging()

ModuleNotFoundError: No module named 'yfinance'

## Config

In [ ]:
YEAR = 2025
WINDOW = 64
BATCH_SIZE = 32
N_EPOCHS = 100
MODEL_TYPE = "lstm"  # "lstm" or "transformer"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Sentiment processing is expensive — cached after first run.
# Delete this file to reprocess.
SENTIMENT_CACHE_PATH = Path("../data/sentiment_daily.pkl")

symbols = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "TSLA", "AVGO", "PEP", "COST",
]

print(f"Device: {DEVICE}")

NameError: name 'torch' is not defined

## Market Data

Technical indicators (SMA-60, RSI-14, ATR-14, …) require **daily** bars.
If the cache already holds daily data for all symbols, this cell skips the fetch.
If data is missing or is sub-daily (hourly), it re-fetches from Alpaca and
overwrites the cached file.

In [3]:
from datetime import datetime

cache = MarketDataCache()


def _is_daily(df: pd.DataFrame) -> bool:
    """Return True when the median gap between rows is ≥ 20 hours (daily cadence)."""
    if len(df) < 2:
        return False
    median_gap = df.index.to_series().diff().median()
    return median_gap >= pd.Timedelta("20h")


alpaca = AlpacaSource()
start = datetime(YEAR, 1, 1)
end   = datetime(YEAR, 12, 31)

for symbol in symbols:
    df = cache.load(symbol, YEAR)
    if _is_daily(df):
        print(f"{symbol}: daily data already cached ({len(df)} bars)")
        continue

    reason = "sub-daily frequency" if df is not None and not df.empty else "not in cache"
    print(f"{symbol}: {reason} — fetching daily bars from Alpaca…")
    df = alpaca.fetch_bars(
        symbols=[symbol],
        timeframe="1Day",
        start=start,
        end=end,
    )
    cache.store(symbol, YEAR, df)
    print(f"  stored {len(df)} daily bars")

NameError: name 'symbols' is not defined

## Fundamentals

Fetches 9 metrics (P/E, P/B, ROE, …) from yfinance and stores a dated snapshot
in `FundamentalCache`.  Skipped if a snapshot for each symbol already exists.

In [4]:
fund_cache  = FundamentalCache()
fund_source = FundamentalSource()

fund_dfs: dict[str, pd.DataFrame | None] = {}
for symbol in symbols:
    existing = fund_cache.load_df(symbol)
    if existing is not None and not existing.empty:
        fund_dfs[symbol] = existing
        print(f"{symbol}: {len(existing)} snapshot(s) already cached")
        continue

    print(f"{symbol}: fetching fundamentals from yfinance…")
    data = fund_source.fetch(symbol)
    fund_cache.store(symbol, data)
    fund_dfs[symbol] = fund_cache.load_df(symbol)
    print(f"  stored snapshot: {data}")

NameError: name 'FundamentalCache' is not defined

## Sentiment

Loads articles from `ArticleRepository`, runs BART summarisation + FinBERT encoding,
and aggregates to one row per (ticker, date).  Result is cached to
`data/sentiment_daily.pkl` — delete the file to reprocess.

**If no articles are found** the sentiment branch will receive zero vectors —
the model still trains on technical + fundamental features only.
Populate `ArticleRepository` via `NewsPipeline` or `KaggleImporter` to enable
the full sentiment signal.

In [5]:
if SENTIMENT_CACHE_PATH.exists():
    sentiment_df = pd.read_pickle(SENTIMENT_CACHE_PATH)
    print(f"Loaded cached sentiment: {len(sentiment_df)} rows")
else:
    repo = ArticleRepository()
    symbol_set = set(symbols)

    ticker_articles: dict[str, list] = {s: [] for s in symbols}
    for month in range(1, 13):
        month_df = repo.read_month(YEAR, month)
        if month_df.empty:
            continue
        for _, row in month_df.iterrows():
            for ticker in (row.get("tickers") or []):
                if ticker in symbol_set:
                    ticker_articles[ticker].append(row.to_dict())

    counts = {t: len(a) for t, a in ticker_articles.items()}
    total  = sum(counts.values())
    print(f"Articles found per ticker: {counts}")
    print(f"Total: {total}")

    if total == 0:
        print(
            "\nNo articles found — sentiment features will be zero vectors.\n"
            "Run NewsPipeline or KaggleImporter to populate ArticleRepository."
        )
        sentiment_df = None
    else:
        pipeline = SentimentPipeline(device=DEVICE)
        sentiment_df = pipeline.process_ticker_articles(ticker_articles)
        SENTIMENT_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        pd.to_pickle(sentiment_df, SENTIMENT_CACHE_PATH)
        print(f"Processed and cached: {len(sentiment_df)} rows")

if sentiment_df is not None:
    display(sentiment_df.head())

NameError: name 'SENTIMENT_CACHE_PATH' is not defined

## Build Datasets

One `FusedDataset` per symbol (chronological order preserved), then pooled
by concatenation.  `build_dataset` raises `RuntimeError` if a symbol has
fewer than 126 daily bars (60 indicator warmup + window=64 + 2 target rows).

In [ ]:
technical = TechnicalFactors()

all_datasets: list[FusedDataset] = []
for symbol in symbols:
    df = cache.load(symbol, YEAR)
    ds = build_dataset(
        df,
        technical,
        sentiment_df=sentiment_df,
        ticker=symbol,
        window=WINDOW,
        fundamental_df=fund_dfs[symbol],
        include_momentum_slope=True,
    )
    all_datasets.append(ds)
    print(f"{symbol}: {len(ds['y'])} windows  label_mean={ds['y'].mean():.3f}")

pooled: FusedDataset = {
    "X_tech":            np.concatenate([d["X_tech"]            for d in all_datasets]),
    "X_sentiment":       np.concatenate([d["X_sentiment"]       for d in all_datasets]),
    "X_fundamental":     np.concatenate([d["X_fundamental"]     for d in all_datasets]),
    "X_sentiment_probs": np.concatenate([d["X_sentiment_probs"] for d in all_datasets]),
    "y":                 np.concatenate([d["y"]                 for d in all_datasets]),
    "dates":             np.concatenate([d["dates"]             for d in all_datasets]),
}

# Sort globally by date so make_loaders' positional split is chronologically correct
# across all symbols (without this, test set = trailing windows of the last symbols only)
sort_idx = np.argsort(pooled["dates"], kind="stable")
pooled = {k: v[sort_idx] for k, v in pooled.items()}

print(f"\nPooled: {len(pooled['y'])} windows total  label_mean={pooled['y'].mean():.3f}")
print(f"X_tech:            {pooled['X_tech'].shape}")
print(f"X_sentiment:       {pooled['X_sentiment'].shape}")
print(f"X_sentiment_probs: {pooled['X_sentiment_probs'].shape}")
print(f"X_fundamental:     {pooled['X_fundamental'].shape}")

## DataLoaders

In [7]:
train_loader, val_loader, test_loader, tech_scaler, fund_scaler = make_loaders(
    pooled,
    test_frac=0.2,
    val_frac=0.1,
    batch_size=BATCH_SIZE,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

NameError: name 'make_loaders' is not defined

## Model

`n_fundamentals` and `n_sentiment_probs` are read from the pooled arrays so
the architecture automatically matches the available features — if fundamentals
or sentiment were missing, the corresponding model branch is disabled.

In [8]:
n_fundamentals    = pooled["X_fundamental"].shape[1]
n_sentiment_probs = pooled["X_sentiment_probs"].shape[2]

if MODEL_TYPE == "lstm":
    model = SentimentLSTM(
        n_factors=16,
        sentiment_dim=768,
        hidden_size=32,
        num_layers=2,
        dropout=0.2,
        n_fundamentals=n_fundamentals,
        n_sentiment_probs=n_sentiment_probs,
    ).to(DEVICE)
else:
    model = SentimentTransformer(
        n_factors=16,
        sentiment_dim=768,
        d_model=64,
        nhead=4,
        n_layers=6,
        dim_feedforward=128,
        dropout=0.2,
        n_fundamentals=n_fundamentals,
        n_sentiment_probs=n_sentiment_probs,
    ).to(DEVICE)

print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

NameError: name 'pooled' is not defined

## Training

In [9]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    patience=15,
    device=DEVICE,
)

print(f"Best epoch: {history['best_epoch']}  |  Best val AUC: {history['best_val_auc']:.4f}")

NameError: name 'train_model' is not defined

## Evaluation

In [10]:
result = bootstrap_evaluate(model, test_loader, device=DEVICE, n_bootstrap=1000, seed=42)

print(f"AUC      : {result['auc_mean']:.3f}  [{result['auc_ci_low']:.3f}, {result['auc_ci_high']:.3f}]")
print(f"Accuracy : {result['accuracy_mean']:.3f}  [{result['accuracy_ci_low']:.3f}, {result['accuracy_ci_high']:.3f}]")
print(f"Precision: {result['precision_mean']:.3f}  [{result['precision_ci_low']:.3f}, {result['precision_ci_high']:.3f}]")
print(f"Recall   : {result['recall_mean']:.3f}  [{result['recall_ci_low']:.3f}, {result['recall_ci_high']:.3f}]")

NameError: name 'bootstrap_evaluate' is not defined

## Save Model

In [11]:
torch.save(model.state_dict(), f"universal_{MODEL_TYPE}.pt")
print(f"Saved to universal_{MODEL_TYPE}.pt")

NameError: name 'model' is not defined

## Live Inference

Take the last 64-day window for a single stock and predict
P(close[t+2] > close[t-1]).

In [ ]:
infer_symbol = "AAPL"

df_infer = cache.load(infer_symbol, YEAR)
ds_infer = build_dataset(
    df_infer,
    technical,
    sentiment_df=sentiment_df,
    ticker=infer_symbol,
    window=WINDOW,
    fundamental_df=fund_dfs[infer_symbol],
    include_momentum_slope=True,
)

# Take the last window and apply the fitted scalers
X_tech_last  = ds_infer["X_tech"][[-1]].copy()           # (1, window, 16)
X_sent_last  = ds_infer["X_sentiment"][[-1]]              # (1, window, 768)
X_fund_last  = ds_infer["X_fundamental"][[-1]].copy()     # (1, n_fund)
X_sprob_last = ds_infer["X_sentiment_probs"][[-1]]        # (1, window, 3)

n, w, f = X_tech_last.shape
X_tech_last = tech_scaler.transform(X_tech_last.reshape(-1, f)).reshape(n, w, f).astype(np.float32)
if fund_scaler is not None:
    X_fund_last = fund_scaler.transform(X_fund_last).astype(np.float32)

model.eval()
with torch.no_grad():
    logit = model(
        torch.tensor(X_tech_last).to(DEVICE),
        torch.tensor(X_sent_last).to(DEVICE),
        fundamentals=torch.tensor(X_fund_last).to(DEVICE),
        sentiment_probs=torch.tensor(X_sprob_last).to(DEVICE),
    )
    prob_up = torch.sigmoid(logit).item()

last_close = df_infer["close"].iloc[-1]
print(f"{infer_symbol} last close  : ${last_close:.2f}")
print(f"P(price goes up) : {prob_up:.4f}")
print(f"Signal           : {'BUY' if prob_up > 0.5 else 'HOLD/SELL'}")